In [6]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import pennylane as qml
from flax import nnx
from functools import partial
import time
import jax
import jax.numpy as jnp
import netket as nk
import optax
from scipy.sparse.linalg import eigsh
from jax import flatten_util
import pickle
import sys
sys.path.append('../')
from H2_631G import SINGLE_SIZE, ha, hi_ext, ext_edges, K, Hatree_Fock,hi,E_fcis,hi,single_rule,g,hf_ground_energy
import logging

jax.config.update("jax_enable_x64", True)

In [2]:
import flax.nnx as nnx
class SingleStateAnsatz_Amplitude(nnx.Module):
    """单态 Ansatz：适配费米子系统的实数值 FFNN"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=float)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=float)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=float)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)
    
class SingleStateAnsatz_Phase(nnx.Module):
    """单态 Ansatz：适配费米子系统的相位 FFNN"""
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.n_spin_orbitals = n_spin_orbitals
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=float)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=float)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=float)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = nnx.tanh(self.linear1(x))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

    
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim: int = 16, *, rngs: nnx.Rngs):
        super().__init__()
        self.real = SingleStateAnsatz_Amplitude(n_spin_orbitals, hidden_dim, rngs=rngs)
        self.phase = SingleStateAnsatz_Phase(n_spin_orbitals, hidden_dim, rngs=rngs)
    

    def __call__(self, x: jax.Array) -> jax.Array:
        logA = self.real(x)    # log(A)
        phi = self.phase(x)    # phase φ
        logpsi = logA + 1j * phi
        return logpsi


In [3]:
# ===================== 4. 包装模型为 machine 函数 =====================
def create_machine(model: nnx.Module):
    """将 Flax NNX 模型包装为 NetKet 风格的 machine 函数"""
    graphdef, state = nnx.split(model)
    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)
    return machine, graphdef, state

# ===================== 局部能量（不变，无梯度） =====================
@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    """
    计算局部能量 E_loc(σ) = Σ_η H(σ→η) ψ(η)/ψ(σ)
    这对应 NetKet 的 local_value_kernel
    """
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    logpsi_eta = machine(params, eta)
    logpsi_sigma = jnp.expand_dims(logpsi_sigma, -1)
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma), axis=-1)

def statistics(x):
    """计算样本统计量"""
    mean = jnp.mean(x)
    var = jnp.var(x)
    return mean, jnp.sqrt(var / x.shape[0])

# ===================== force-based梯度：增加holomorphic静态参数 =====================
@partial(jax.jit, static_argnames=("machine", "holomorphic"))
def forces_expect_hermitian(machine, params, sigma, holomorphic):
    """
    ∇⟨E⟩ = ⟨(E_loc - ⟨E⟩) ∇log ψ⟩
    holomorphic:
        True: 复全纯函数（复参数网络）
        False: 实参数网络，振幅相位分离，必须False
    """
    # 1. 局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    O_mean, O_std = statistics(O_loc)
    O_centered = O_loc - O_mean

    # 2. 每个样本求 ∇logψ
    def log_psi_single(p, s):
        return machine(p, s)

    def compute_grad_for_sample(s, holo_flag):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=holo_flag)(params)

    grad_matrix = jax.vmap(compute_grad_for_sample, in_axes=(0, None))(sigma, holomorphic)

    # 3. force梯度
    def weight_and_mean(grad_component):
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    return O_mean, O_std, grad

# ===================== QGT 计算：同样增加holomorphic参数 =====================
@partial(jax.jit, static_argnames=("machine", "holomorphic"))
def compute_qgt(machine, params, sigma, diag_shift=0.1, holomorphic=False):
    """
    QGT / F矩阵
    holomorphic=False：实参数振幅相位分离网络（你的情况）
    holomorphic=True：复全纯波函数
    """
    n_samples = sigma.shape[0]
    def log_psi_single(p, s):
        return machine(p, s)

    def compute_grad_for_sample(s, holo_flag):
        return jax.grad(lambda p: log_psi_single(p, s), holomorphic=holo_flag)(params)

    grad_matrix = jax.vmap(compute_grad_for_sample, in_axes=(0, None))(sigma, holomorphic)

    # 展平pytree
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)

    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)
    grad_centered = grad_flat - grad_mean

    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    return qgt_reg, unravel_fn


In [4]:
@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """
    实参数网络，输出复logψ，非全纯，使用vjp计算∇logψ，避开jax.grad限制
    """
    # 1. 局部能量
    O_loc = compute_local_energies(machine, params, sigma)
    O_mean, O_std = statistics(O_loc)
    O_centered = O_loc - O_mean

    def log_psi_single(p, s):
        return machine(p, s)

    # ===== 替换掉原来 jax.grad 的部分，vjp版本 =====
    def compute_grad_for_sample(s):
        def f(p):
            return log_psi_single(p, s)
        val, vjp_fun = jax.vjp(f, params)
        # Wirtinger梯度：∇_θ f = ∂f/∂θ
        grad_real, = vjp_fun(1.0 + 0.0j)
        grad_imag, = vjp_fun(0.0 + 1.0j)
        grad = jax.tree.map(lambda r, i: r + 1j*i, grad_real, grad_imag)
        return grad

    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)

    # force梯度公式 ⟨(E_loc - E_mean) conj(∇logψ)⟩
    def weight_and_mean(grad_component):
        weights = O_centered.reshape((O_centered.shape[0],) + (1,) * (grad_component.ndim - 1))
        return jnp.mean(weights * jnp.conj(grad_component), axis=0)
    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    return O_mean, O_std, grad


#@partial(jax.jit, static_argnames=("machine",))
def compute_qgt(machine, params, sigma, diag_shift=0.1):
    n_samples = sigma.shape[0]
    def log_psi_single(p, s):
        return machine(p, s)

    def compute_grad_for_sample(s):
        def f(p):
            return log_psi_single(p, s)
        val, vjp_fun = jax.vjp(f, params)
        grad_real, = vjp_fun(1.0 + 0.0j)
        grad_imag, = vjp_fun(0.0 + 1.0j)
        grad = jax.tree.map(lambda r, i: r + 1j*i, grad_real, grad_imag)
        return grad

    grad_matrix = jax.vmap(compute_grad_for_sample)(sigma)

    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)

    grad_mean = jnp.mean(grad_flat, axis=0, keepdims=True)
    grad_centered = grad_flat - grad_mean
    qgt = (1.0 / n_samples) * jnp.conj(grad_centered).T @ grad_centered
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    return qgt_reg, unravel_fn


In [5]:
model = SingleStateAnsatz(n_spin_orbitals=SINGLE_SIZE,hidden_dim=SINGLE_SIZE*2+4,rngs=nnx.Rngs(11))
sampler = nk.sampler.MetropolisSampler(hi, rule=single_rule, n_chains=200, sweep_size=20)
optimizer = nk.optimizer.Sgd(learning_rate=0.1)
machine, graphdef, params = create_machine(model)
sampler_state = sampler.init_state(machine, params, seed=1)
optimizer = optax.sgd(learning_rate=0.01)  # 学习率 0.01
opt_state = optimizer.init(params)

# 训练参数
N_ITER = 400  # 迭代次数
N_SAMPLES = 1008  # 样本数

# ===================== 7. 训练循环 =====================
print("\n" + "="*60)
print("开始纯 JAX VMC 训练 (自然梯度下降法)")
print("="*60)

# 用于记录训练历史
history = {
    'step': [],
    'energy': [],
    'energy_std': [],
    'error': []
}

for step in range(N_ITER):
    # 1. 采样
    sampler_state = sampler.reset(machine,params,sampler_state)
    
    samples, sampler_state = sampler.sample(
        machine, params, state=sampler_state, 
        chain_length=20
    )
    samples = samples.reshape(-1, hi.size)
    
    # 2. 计算 force-based 能量和梯度
    energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)
    #grad = jax.tree_map(lambda x: x*2, grad)
    grad = jax.tree_util.tree_map(lambda x: x * 2, grad)
    #qgt_reg, unravel_fn = compute_qgt(machine,params,vstate.samples.reshape(-1,4),0.001)
    qgt_reg, qgt_unravel_fun = compute_qgt(machine, params, samples, diag_shift=0.01) 
    grad_flat , grad_unravel_fn = flatten_util.ravel_pytree(grad)
  
    #自然梯度 natural-gradient = S^{-1} * grad
    natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
    natural_grad = grad_unravel_fn(natural_grad)
    grad = natural_grad
        
    # 4. 更新参数（自然梯度下降）
    updates, opt_state = optimizer.update(grad, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    # 5. 记录历史
    if step % 50 == 0 or step == N_ITER - 1:
        error = jnp.abs(energy.real - E_fcis[0])
        history['step'].append(step)
        history['energy'].append(float(energy.real))
        history['energy_std'].append(float(energy_std))
        history['error'].append(float(error))
        print(f"Step {step:3d} | E: {energy.real:.8f} ± {energy_std:.6f} | FCI: {E_fcis[0]:.8f} | Error: {error:.6f}")

# 最终结果
final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
final_error = jnp.abs(final_energy.real - E_fcis[0])
print("\n" + "="*60)
print(f"训练完成!")
print(f"最终能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
print(f"绝对误差：{final_error:.6f} Ha")
print(f"相对误差：{final_error / jnp.abs(E_fcis[0]) * 100:.4f}%")
print("="*60)




开始纯 JAX VMC 训练 (自然梯度下降法)
Step   0 | E: 0.73079585 ± 0.012667 | FCI: -1.05434745 | Error: 1.785143
Step  50 | E: -0.99693977 ± 0.000380 | FCI: -1.05434745 | Error: 0.057408
Step 100 | E: -0.99747916 ± 0.000000 | FCI: -1.05434745 | Error: 0.056868
Step 150 | E: -0.99748087 ± 0.000000 | FCI: -1.05434745 | Error: 0.056867
Step 200 | E: -0.99733419 ± 0.000148 | FCI: -1.05434745 | Error: 0.057013
Step 250 | E: -0.99748322 ± 0.000000 | FCI: -1.05434745 | Error: 0.056864
Step 300 | E: -0.99748382 ± 0.000000 | FCI: -1.05434745 | Error: 0.056864
Step 350 | E: -0.99748423 ± 0.000000 | FCI: -1.05434745 | Error: 0.056863
Step 399 | E: -0.99742023 ± 0.000065 | FCI: -1.05434745 | Error: 0.056927

训练完成!
最终能量：-0.99742018 ± 0.000065 Ha
FCI 基准：-1.05434745 Ha
绝对误差：0.056927 Ha
相对误差：5.3993%


In [7]:
hf_ground_energy

np.float64(-0.9974972943283582)